# Retail Prediction Assignment
kind of messy but works

## 1. Date Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = pd.read_csv("q3_retail_promotions.csv")

data['transaction_date'] = pd.to_datetime(data['transaction_date'])

data['year'] = data['transaction_date'].dt.year
data['month'] = data['transaction_date'].dt.month
data['day_of_week'] = data['transaction_date'].dt.dayofweek

data['is_month_end'] = 0
data.loc[data['transaction_date'].dt.day >= 25, 'is_month_end'] = 1

data.head()

## 2. Temporal Split

In [ ]:
data = data.sort_values("transaction_date")

cut = int(len(data)*0.8)

train_df = data.iloc[:cut]
test_df = data.iloc[cut:]

X_tr = train_df.drop(['items_sold','transaction_date'], axis=1)
y_tr = train_df['items_sold']

X_te = test_df.drop(['items_sold','transaction_date'], axis=1)
y_te = test_df['items_sold']

print(X_tr.shape, X_te.shape)

Random splitting is not good here because the data is time-based. If we randomly split, some future data might go into training and past into testing, which is unrealistic. In real life we only know the past when predicting the future, so this way is better.

## 3. Preprocessing

In [ ]:
cattt = ['promotion_type','location_type','store_size']
numzz = [c for c in X_tr.columns if c not in cattt]

ct = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown='ignore'), cattt),
    ('scaler', StandardScaler(), numzz)
])

ct

## 4. Models

In [ ]:
pipe1 = Pipeline([
    ('prep', ct),
    ('m', LinearRegression())
])

pipe2 = Pipeline([
    ('prep', ct),
    ('m', RandomForestRegressor(n_estimators=100, random_state=42))
])

pipe1.fit(X_tr, y_tr)
pipe2.fit(X_tr, y_tr)

pred1 = pipe1.predict(X_te)
pred2 = pipe2.predict(X_te)

rmse1 = np.sqrt(mean_squared_error(y_te, pred1))
mae1 = mean_absolute_error(y_te, pred1)

rmse2 = np.sqrt(mean_squared_error(y_te, pred2))
mae2 = mean_absolute_error(y_te, pred2)

print("Linear RMSE", rmse1)
print("Linear MAE", mae1)

print("RF RMSE", rmse2)
print("RF MAE", mae2)

In [ ]:
plt.figure()
plt.scatter(y_te, pred1)
plt.plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()])
plt.title("linear plot")
plt.show()

plt.figure()
plt.scatter(y_te, pred2)
plt.plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()])
plt.title("rf plot")
plt.show()

In [ ]:
ohe = pipe2.named_steps['prep'].named_transformers_['ohe']
names1 = ohe.get_feature_names_out(cattt)

allnames = list(names1) + numzz

imps = pipe2.named_steps['m'].feature_importances_

df_imp = pd.DataFrame({'f': allnames, 'imp': imps})
df_imp = df_imp.sort_values(by='imp', ascending=False)

df_imp.head(5)